In [ ]:
import os
os.environ["GROQ_API_KEY"] = "api-key"

## Install libraries

In [97]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

In [8]:
pip install youtube-transcript-api==0.6.1

  Attempting uninstall: youtube-transcript-api
    Found existing installation: youtube-transcript-api 1.2.4
    Uninstalling youtube-transcript-api-1.2.4:
      Successfully uninstalled youtube-transcript-api-1.2.4


In [98]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [101]:
import subprocess
subprocess.run(["pip", "install", "yt-dlp"], check=True)

CompletedProcess(args=['pip', 'install', 'yt-dlp'], returncode=0)

In [102]:
import subprocess, json

result = subprocess.run(
    ["yt-dlp", "--write-auto-sub", "--sub-lang", "en",
     "--skip-download", "--sub-format", "json3",
     "-o", "/content/transcript", f"https://www.youtube.com/watch?v=zfiSAzpy9NM"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)  # check for errors here

[youtube] Extracting URL: https://www.youtube.com/watch?v=zfiSAzpy9NM
[youtube] zfiSAzpy9NM: Downloading webpage
[youtube] zfiSAzpy9NM: Downloading android vr player API JSON
[info] zfiSAzpy9NM: Downloading subtitles: en
[info] zfiSAzpy9NM: Downloading 1 format(s): 401+251
[info] Writing video subtitles to: /content/transcript.en.json3
[download] Destination: /content/transcript.en.json3

[download]    1.00KiB at  542.11KiB/s (00:00:00)
[download]    3.00KiB at  565.80KiB/s (00:00:00)
[download]    7.00KiB at    1.15MiB/s (00:00:00)
[download]   15.00KiB at    2.23MiB/s (00:00:00)
[download]   31.00KiB at    3.75MiB/s (00:00:00)
[download]   63.00KiB at    6.98MiB/s (00:00:00)
[download]  127.00KiB at   12.70MiB/s (00:00:00)
[download]  255.00KiB at   22.17MiB/s (00:00:00)
[download]  349.07KiB at   27.35MiB/s (00:00:00)
[download] 100% of  349.07KiB in 00:00:00 at 3.38MiB/s




In [103]:
def convert_json3_to_transcript(json3_path):
    with open(json3_path) as f:
        data = json.load(f)

    transcript = []
    for event in data["events"]:
        if "segs" not in event or event.get("aAppend") == 1:
            continue

        # Join segs and remove newlines
        text = "".join(seg.get("utf8", "") for seg in event["segs"])
        text = text.replace("\n", " ").strip()

        if not text:
            continue

        transcript.append({
            "text": text,
            "start": round(event["tStartMs"] / 1000, 3),
            "duration": round(event["dDurationMs"] / 1000, 3)
        })
    return transcript

transcript_list = convert_json3_to_transcript("/content/transcript.en.json3")

# Flat plain text
transcript_text = " ".join(chunk["text"] for chunk in transcript_list)
print(transcript_text[:500])

I will give you a very simple explanation of convolutional neural network without using much mathematics such that even a high school student can understand it easily let's say you want the computer to recognize the handwritten digit n the way computer looks at this is as a grid of numbers here I'm using minus one and one in reality it will use RGB numbers from 0 to 255 the issue with this presentation is that this is too much hardcoded if you have a little shift in digit 9 for example nine here


In [104]:
transcript_list

[{'text': 'I will give you a very simple', 'start': 0.04, 'duration': 4.0},
 {'text': 'explanation of convolutional neural',
  'start': 1.64,
  'duration': 5.04},
 {'text': 'network without using much mathematics',
  'start': 4.04,
  'duration': 5.16},
 {'text': 'such that even a high school student can',
  'start': 6.68,
  'duration': 4.4},
 {'text': "understand it easily let's say you want",
  'start': 9.2,
  'duration': 4.08},
 {'text': 'the computer to recognize the', 'start': 11.08, 'duration': 4.16},
 {'text': 'handwritten digit', 'start': 13.28, 'duration': 5.72},
 {'text': 'n the way computer looks at this is as a',
  'start': 15.24,
  'duration': 6.84},
 {'text': "grid of numbers here I'm using minus one",
  'start': 19.0,
  'duration': 6.4},
 {'text': 'and one in reality it will use RGB',
  'start': 22.08,
  'duration': 6.599},
 {'text': 'numbers from 0 to', 'start': 25.4, 'duration': 6.4},
 {'text': '255 the issue with this presentation is',
  'start': 28.679,
  'duration': 

In [105]:
transcript

"in \n situation \n extent \n Muslims \n create \n will \n Armenia \n Corridor \n Gaza \n what \n that \n Palestinians \n cease \n this \n hatred \n these \n likely \n direction \n it \n on \n people \n may \n would \n there's \n as \n andout \n something \n being \n warfare \n China \n they \n look \n have \n weapons \n enough \n world \n tomorrow \n geopolitics \n [Music] \n abijit \n guest \n our \n but \n and \n you \n much \n back \n here \n of \n but \n conversations \n enjoy \n that \n straight \n watch \n educational \n perspective \n little \n Israel \n Israel \n East \n there \n crisis \n in \n years \n always \n begins \n date \n massive \n massive \n cut \n the \n through \n thousand \n clearly \n rights \n call \n off \n idea \n intelligence \n signals \n of \n Israel's \n almost \n of \n Israel's \n the \n is \n reinforment \n enough \n then \n launches \n the \n people \n crisis \n being \n thing \n magnitude \n coming \n situation \n the \n which \n been \n devastated \

In [106]:
transcript_text

"I will give you a very simple explanation of convolutional neural network without using much mathematics such that even a high school student can understand it easily let's say you want the computer to recognize the handwritten digit n the way computer looks at this is as a grid of numbers here I'm using minus one and one in reality it will use RGB numbers from 0 to 255 the issue with this presentation is that this is too much hardcoded if you have a little shift in digit 9 for example nine here was in the middle but in this case it is in the left and the representation of numbers just changes it doesn't matches match with our original number grid and computer will not be able to recognize that this is number nine there could be a variation since it is a and return return digit there could be variation in how you write it which will change the two-dimensional representation of numbers and again you will not be able to match it with the original grid so we use artificial neural network

## Step 1b - Indexing (Text Splitting)

In [113]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.create_documents([transcript_text])

In [114]:
len(chunks)

26

In [115]:
chunks[6]

Document(metadata={}, page_content='we have three filters the first one is the head which is a loopy circle pattern in the middle you have vertical line in the end you have diagonal filter so we take our original image and we will apply a convolution operation or a filter operation so here I have a loopy circle pattern or a head filter this filter right here the way convolution operation works is you take 3x3 grid from your original image and multiply individual numbers with this filter so this minus one is Multiplied with this one this one is Multiplied with this one and so on in the end you get a result and then you find the average which is divide by 9 because there are total nine numbers and whatever number you get you put it here now this particular thing is called a feature map so by doing this convolution operation')

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
from google import genai

client = genai.Client(api_key="api-key")

# Document objects use .page_content not ["text"]
texts = [chunk.page_content for chunk in chunks]

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts
)

embeddings = [e.values for e in result.embeddings]
print(f"Got {len(embeddings)} embeddings, dimension: {len(embeddings[0])}")

Got 26 embeddings, dimension: 3072


In [55]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.5 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# This is the correct embedding object for LangChain
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key="api-key"
)

# Now pass the model object, not the embeddings list
vector_store = FAISS.from_documents(chunks, embedding_model)
print("Vector store created!")

Vector store created!


In [118]:
vector_store.index_to_docstore_id

{0: '302a71c1-839f-4e53-a370-24f9b06a1146',
 1: '5187d3c6-af9f-49b7-8f71-e5594886e68c',
 2: '42fa9d51-bdc1-4e43-ad57-d17944696fe5',
 3: 'eac61adc-c2df-4b7b-bc7a-c9cc32f40d89',
 4: '88d1cd78-00d6-4a4b-8203-ef36ea7bf02f',
 5: 'bb66b63c-fcb7-4f1e-8967-5758d6abc78e',
 6: '2226ef0a-0ab6-48f8-8309-53255498da24',
 7: '1703ea4f-6ba8-43bc-a927-cb4ab5c881fc',
 8: '2b8c0fef-8d5a-40f6-87ac-1ef378739d2c',
 9: '30ff6445-fe74-4876-bff8-68cfd8275747',
 10: 'bc6f9b2e-caa6-4bb8-a2b9-bb578af6e432',
 11: 'baad4031-8634-4fb4-bd11-69997433c55b',
 12: '2a776195-1109-4147-aa27-eec4a5ef9965',
 13: '0e3a1964-1353-455f-a47c-40d1d4a1638f',
 14: '321f1cae-66a4-4df4-a09e-e3a79efa9a08',
 15: '0585348a-e937-476f-85a3-4e74618946da',
 16: 'c29afc10-a206-4aa5-b0e7-085e4dd2ad52',
 17: '754e2a4d-851d-4c05-a2fc-dcb7f435a202',
 18: '58924d54-4b51-4a66-8b5b-b0171e691c2b',
 19: '89391b18-9dfd-4dc8-8143-389f29f24052',
 20: '537bd77b-6e4f-42e6-894e-4acc3839089e',
 21: '43f6cb02-ea5d-4696-ae7d-fb6c4cbf9dc0',
 22: 'f4db861c-dea5-

In [119]:
vector_store.get_by_ids(['da6d5f90-f24f-4332-8ec9-bcfbf59cfdb1'])

[Document(id='da6d5f90-f24f-4332-8ec9-bcfbf59cfdb1', metadata={}, page_content="can learn machine learning in a very very easy to understand manner then gradually in this playlist I try to cover data science and machine learning projects as well I'm continuing my deep learning tutorial Series right now and my goal is to finish all the topics in deep learning including convolutional neural networks RNN language models and so on so please stay tuned uh watch my videos and if you have any comments or feedback please let me know in the video comment below")]

## Step 2 - Retrieval

In [120]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [121]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7d4a66b669c0>, search_kwargs={'k': 4})

In [122]:
retriever.invoke('What is neural network')

[Document(id='88d1cd78-00d6-4a4b-8203-ef36ea7bf02f', metadata={}, page_content="try to see how we as humans recognize any image so easily when we look at koala's image we look at the little features like this round eyes this black prominent flat nose this fluffy ears and we detect these features one by one in our brain there are different set of neurons working on these different features and they're firing they're saying yeah I found koala's ears yes I found koala's nose and so on then these neurons are connected to another set of neurons which will aggregate the results it will say if in the image you are seeing koala's eye nose and ears it means there is a koala's face in the image similarly if there is koala's hands and legs it means there is koala's body and there are different set of neurons which are connected to these neurons which will again aggregate the"),
 Document(id='42fa9d51-bdc1-4e43-ad57-d17944696fe5', metadata={}, page_content="number and then we build a neural networ

## Step 3 - Augmentation

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key="api-key"
)

In [124]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [130]:
question          = "is the topic of pooling discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [131]:
retrieved_docs

[Document(id='754e2a4d-851d-4c05-a2fc-dcb7f435a202', metadata={}, page_content="are detecting the loopy pattern at the top so max pulling along with convolution helps you with uh position invariant feature detection doesn't matter where your eyes or ears are in the image it will detect that feature for you there is average pulling also instead of Max you just make an average see pi and 1 6 and 2 8 8 and 8 16 16 4 is 4 so but Max pulling is more generally used but sometimes people use average pooling also so benefits of pooling number one obvious it's reducing your dimension and computation the second benefit it reduce overfitting because there are less parameters and the third one is model is variant tolerant towards variation and Distortion because if there is a distortion and if you're picking just a maximum number you are capturing the main feature and you are"),
 Document(id='0585348a-e937-476f-85a3-4e74618946da', metadata={}, page_content="people don't use padding so they reduce t

In [132]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"are detecting the loopy pattern at the top so max pulling along with convolution helps you with uh position invariant feature detection doesn't matter where your eyes or ears are in the image it will detect that feature for you there is average pulling also instead of Max you just make an average see pi and 1 6 and 2 8 8 and 8 16 16 4 is 4 so but Max pulling is more generally used but sometimes people use average pooling also so benefits of pooling number one obvious it's reducing your dimension and computation the second benefit it reduce overfitting because there are less parameters and the third one is model is variant tolerant towards variation and Distortion because if there is a distortion and if you're picking just a maximum number you are capturing the main feature and you are\n\npeople don't use padding so they reduce the image size but only little bit so pulling is used to reduce the size the main purpose of pulling is to reduce the dimension so that my computer doesn't get 

In [133]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [134]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      are detecting the loopy pattern at the top so max pulling along with convolution helps you with uh position invariant feature detection doesn't matter where your eyes or ears are in the image it will detect that feature for you there is average pulling also instead of Max you just make an average see pi and 1 6 and 2 8 8 and 8 16 16 4 is 4 so but Max pulling is more generally used but sometimes people use average pooling also so benefits of pooling number one obvious it's reducing your dimension and computation the second benefit it reduce overfitting because there are less parameters and the third one is model is variant tolerant towards variation and Distortion because if there is a distortion and if you're picking just a maximum number you are capturing the main feature and you are\n\npeople don

## Step 4 - Generation

In [135]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of pooling is discussed in this video. 

The discussion includes:
1. The purpose of pooling: to reduce dimension and computation.
2. Types of pooling: Max pooling and Average pooling, with Max pooling being more commonly used.
3. Benefits of pooling: 
   - Reduces dimension and computation.
   - Reduces overfitting by having fewer parameters.
   - Makes the model more tolerant to variations and distortions.
4. How Max pooling works: taking a window (e.g., 2x2), picking the maximum number, and putting it in the new feature map, resulting in a reduced feature map size.
5. The effect of stride on Max pooling: a stride of 2 means moving 2 pixels after processing a window, while a stride of 1 means moving 1 pixel.
6. The role of pooling in helping with position-invariant feature detection, especially when used with convolution.


## Building a Chain

In [147]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [148]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [149]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [150]:
parallel_chain.invoke('what is pooling')

{'context': "are detecting the loopy pattern at the top so max pulling along with convolution helps you with uh position invariant feature detection doesn't matter where your eyes or ears are in the image it will detect that feature for you there is average pulling also instead of Max you just make an average see pi and 1 6 and 2 8 8 and 8 16 16 4 is 4 so but Max pulling is more generally used but sometimes people use average pooling also so benefits of pooling number one obvious it's reducing your dimension and computation the second benefit it reduce overfitting because there are less parameters and the third one is model is variant tolerant towards variation and Distortion because if there is a distortion and if you're picking just a maximum number you are capturing the main feature and you are\n\npeople don't use padding so they reduce the image size but only little bit so pulling is used to reduce the size the main purpose of pulling is to reduce the dimension so that my computer 

In [151]:
parser = StrOutputParser()

In [152]:
main_chain = parallel_chain | prompt | llm | parser

In [153]:
main_chain.invoke('what is pooling')

'Pooling is a process used to reduce the dimension of an image or feature map. It involves taking a window of a certain size (e.g. 2x2) and selecting the maximum (max pooling) or average (average pooling) value from that window to create a new, smaller feature map. This helps to reduce computation, prevent overfitting, and make the model more tolerant to variations and distortions.'